In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import json

from src.config.db import get_connection
from src.config.llm import llm

conn = get_connection()

In [3]:
document_id = "DOC000002"
BATCH_SIZE = 5
conn = get_connection()
with conn.cursor() as cur:

    cur.execute(
        """
        SELECT
            id,
            "pageNumber",
            content
        FROM "Page"
        WHERE "documentId"=%s
        ORDER BY "pageNumber"
        """,
        (document_id,),
    )

    pages = cur.fetchall()

print(f"Fetched {len(pages)} pages")

Fetched 290 pages


# PROMPT

In [4]:
def batches(items, size):

    for i in range(0, len(items), size):
        yield items[i:i+size]

PROMPT = """
You are building page metadata for a Vectorless RAG indexing system.

You will receive multiple pages.

For EACH page generate metadata.

Return ONLY valid JSON.

Return an array.

Schema:

[
  {{
    "pageNumber": 1,
    "title": "",
    "summary": "",
    "keywords": [],
    "entities": [],
    "topics": [],
    "pageType": "cover|toc|content|appendix|references",
    "containsTable": false,
    "containsFigure": false
  }}
]

Pages

{pages}
"""

# Generate Metadata

In [5]:
from email.mime import text


def generate_metadata(batch):

    page_text = []

    for page in batch:

        page_text.append(
            f"""
Page Number: {page[1]}

{page[2]}
"""
        )

    prompt = PROMPT.format(
        pages="\n\n".join(page_text)
    )

    response = llm.invoke(prompt)

    text = response.content.strip()

    if text.startswith("```json"):
        text = text[7:]

    if text.endswith("```"):
        text = text[:-3]


    print("=" * 80)
    print(text)
    print("=" * 80)
    return json.loads(text)
all_metadata = []

for batch in batches(pages, BATCH_SIZE):

    metadata = generate_metadata(batch)

    all_metadata.extend(metadata)

print(f"Generated metadata for {len(all_metadata)} pages")



[
  {
    "pageNumber": 1,
    "title": "Microsoft Fabric Documentation for Admins",
    "summary": "Introduction to Microsoft Fabric admin settings, options, and tools",
    "keywords": ["Microsoft Fabric", "admin settings", "admin tools"],
    "entities": ["Microsoft", "Fabric"],
    "topics": ["administration", "settings", "tools"],
    "pageType": "cover",
    "containsTable": false,
    "containsFigure": false
  },
  {
    "pageNumber": 2,
    "title": "Configuring Notifications and Monitoring",
    "summary": "Configure notifications, set up metadata scanning, and enable content certification",
    "keywords": ["notifications", "metadata scanning", "content certification"],
    "entities": ["Microsoft Fabric"],
    "topics": ["configuration", "monitoring"],
    "pageType": "content",
    "containsTable": false,
    "containsFigure": false
  },
  {
    "pageNumber": 3,
    "title": "Administration Overview",
    "summary": "Overview of Microsoft Fabric administration, including l

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kwxmsypje8evkkkq95tjh1k2` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99427, Requested 1348. Please try again in 11m9.6s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

# Store Metadata

In [ ]:
with conn.cursor() as cur:

    for meta in all_metadata:

        cur.execute(
            """
            UPDATE "Page"
            SET
                metadata=%s
            WHERE
                "documentId"=%s
                AND
                "pageNumber"=%s
            """,
            (
                json.dumps(meta),
                document_id,
                meta["pageNumber"],
            ),
        )

conn.commit()

print("Metadata Stored Successfully")


Metadata Stored Successfully


# verify store data

In [ ]:
with conn.cursor() as cur:

    cur.execute(
        """
        SELECT
            "pageNumber",
            metadata
        FROM "Page"
        WHERE "documentId"=%s
        ORDER BY "pageNumber"
        """,
        (document_id,),
    )

    rows = cur.fetchall()

rows[:5]

[(1,
  {'title': 'REPORT TO CONGRESS',
   'topics': ['Economy', 'Monetary Policy', 'Financial Stability'],
   'summary': '110th Annual Report of the Board of Governors of the Federal Reserve System',
   'entities': ['Federal Reserve System', 'Board of Governors'],
   'keywords': ['Federal Reserve', 'Annual Report', 'Board of Governors'],
   'pageType': 'cover',
   'pageNumber': 1,
   'containsTable': False,
   'containsFigure': False}),
 (2,
  {'title': '',
   'topics': [],
   'summary': '',
   'entities': [],
   'keywords': [],
   'pageType': 'content',
   'pageNumber': 2,
   'containsTable': False,
   'containsFigure': False}),
 (3,
  {'title': 'Contents',
   'topics': ['Economy', 'Monetary Policy', 'Financial Stability'],
   'summary': 'Table of contents for the 110th Annual Report of the Board of Governors of the Federal Reserve System',
   'entities': ['Federal Reserve System', 'Board of Governors'],
   'keywords': ['Table of Contents', 'Annual Report'],
   'pageType': 'toc',
   '